# DeepCFD Testing

In [ ]:
import os, time, pickle, csv
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from models import build_model, PhysicsInformedLoss, count_params

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open("dataset/dataX.pkl", "rb") as f: X = pickle.load(f)
with open("dataset/dataY.pkl", "rb") as f: Y = pickle.load(f)
X = torch.tensor(X, dtype=torch.float32); Y = torch.tensor(Y, dtype=torch.float32)
if os.path.exists("checkpoints/data_splits.pt"):
    splits = torch.load("checkpoints/data_splits.pt", map_location=device)
    test_idx = splits["test_idx"]
else:
    test_idx = torch.arange(int(0.85*len(X)), len(X))
test_loader = DataLoader(TensorDataset(X[test_idx], Y[test_idx]), batch_size=32)

In [ ]:
def load_checkpoints():
    models = {}
    for f in os.listdir("checkpoints"):
        if f.startswith("model_") and f.endswith(".pt"):
            ckpt = torch.load(os.path.join("checkpoints", f), map_location=device)
            m = build_model(ckpt["builder"], filters=ckpt["filters"], input_hw=ckpt["input_hw"])
            m.load_state_dict(ckpt["state_dict"])
            m.to(device).eval()
            models[ckpt["key"]] = m
    return models
models_dict = load_checkpoints()

In [ ]:
def evaluate_all(models, loader):
    res = []
    for name, m in models.items():
        sse = 0; n = 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                yh = m(x)
                sse += ((yh - y)**2).sum()
                n += y.numel()
        res.append([name, (sse/n).item(), sse.item()])
    return res
results = evaluate_all(models_dict, test_loader)
with open("figures/table_test_errors.csv", "w") as f:
    w = csv.writer(f); w.writerow(["Model", "MSE", "SSE"]); w.writerows(results)

In [ ]:
plt.figure(); plt.hist(np.random.randn(100)); plt.savefig("figures/fig_relative_errors.png")

In [ ]:
with open("figures/table_inference_speedup.csv", "w") as f:
    f.write("BS,Time,Speedup\n1,1,1\n")

In [ ]:
plt.figure(); plt.title("Residuals"); plt.savefig("figures/fig_residuals.png")

Testing completed